# SNIa by Fink_ID — Fink Alert Download + SALT3 Fit

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/IN2P3/CNRS — Université Paris-Saclay
- **creation date** : 2026-06-15
- **last update** : 2026-06-24 : fetch the redshift from Fink and add the ellipse contour plots
## Purpose

Download the Fink LSST alert stream for the single object:

| Field | Value |
|-------|-------|
| `r:diaObjectId` | `739161397740437575` |
| Fink internal ID | `313629129605382159` |

Two data products are retrieved from the Fink API:

- **src** — difference-image alert photometry (`/api/v1/sources`): `psfFlux` in nJy
- **fp**  — forced photometry (`/api/v1/forcedphotometry`): `fpFlux` in nJy

The redshift is looked up from the Fink cross-match columns:

- `f:xm_tns_redshift` — spectroscopic z from TNS (best)
- `f:xm_legacydr8_zphot` — photometric z from Legacy Survey DR8
- `f:xm_mangrove_lum_dist` — MANGROVE luminosity distance

Then the multi-band light curve is fitted with **SALT3** (`salt3-nir` or `salt2-extended`
as a fallback) via `sncosmo`, in two modes:

1. **z fixed** to the best available redshift (TNS spec-z if present, else photo-z)
2. **z free** — two-pass grid scan + refinement (same strategy as `04_fink_tns_sn_fitSNIa_salt2_withredshift_lightcurves.ipynb`)

The Hubble distance modulus μ is derived via the Tripp formula.

### Column conventions
- `r:<col>` — LSST diaSource/diaObject field (prefix = table name, NOT r-band)
- `f:<col>` — Fink-computed field
- Flux unit: **nJy**, AB ZP = 31.4


## 0 — Imports

In [ ]:
import io
import math
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import astropy.table
import requests
import sncosmo
from scipy.integrate import quad
from astropy.time import Time
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Ellipse

from scipy.stats import chi2

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("Imports OK")
print(f"sncosmo version : {sncosmo.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → %matplotlib inline")

## 1 — Configuration

In [ ]:
# ── Target object ─────────────────────────────────────────────────────────────
# Garazi candidate where DIA_OBJECT_ID is the ID in the Butler DP2
# DIA_OBJECT_ID = 739161397740437575  # r:diaObjectId (Rubin)
# FINK_ID = 313629129605382159  # Fink internal objectId (used for forcedphotometry)
# my candidate here DIA_OBJECT_ID is the FINK_ID, the ID in he AP pipeline
DIA_OBJECT_ID = 170226393328648251
FINK_ID = 170226393328648251

# ── Fink LSST API ─────────────────────────────────────────────────────────────
FINK_API = "https://api.lsst.fink-portal.org/api/v1"
API_DELAY = 0.4  # seconds between requests (rate limiting)

# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "NB97_02_SNIaGarazi"
DATA_DIR = Path(f"data_{NB_TAG}")
FIGS_DIR = Path(f"figs_{NB_TAG}")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# ── Photometric system ────────────────────────────────────────────────────────
RUBIN_ZP = 31.4  # AB zeropoint for nJy fluxes
ZPSYS = "ab"

# ── SALT model ────────────────────────────────────────────────────────────────
# Try salt3 first; fall back to salt2-extended if not available
SALT_SOURCES_PRIORITY = ["salt3", "salt2-extended"]
BAND_PREFIX = "lsst"  # sncosmo LSST band names: lsstu, lsstg, ...
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]

# ── Quality cuts for SALT fit ─────────────────────────────────────────────────
SNR_MIN = 3.0  # minimum SNR per point (psfFlux / psfFluxErr)
MIN_DATAPOINTS = 5  # minimum points after quality cuts

# ── Two-pass free-z strategy ──────────────────────────────────────────────────
Z_BOUND_LO = 0.02
Z_BOUND_HI = 1.50
N_GRID_Z = 100
Z_REFINE_HALF = 0.15  # ±half-window around best grid z for free-z fit

# ── Tripp formula nuisance parameters ────────────────────────────────────────
ALPHA = 0.14  # Betoule+ 2014
BETA = 3.14  # Betoule+ 2014
M_B = -19.3  # absolute B-band magnitude

# ── Flat ΛCDM cosmology for Hubble diagram ────────────────────────────────────
H0 = 70.0
OmegaM = 0.30
OmegaL = 0.70

# ── Band colours ─────────────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#1f77b4",  # blue
    "g": "#2ca02c",  # green
    "r": "#d62728",  # red
    "i": "#ff7f0e",  # orange
    "z": "#8c564b",  # brown
    "y": "#9467bd",  # purple
}

print(f"diaObjectId  : {DIA_OBJECT_ID}")
print(f"Fink ID      : {FINK_ID}")
print(f"Data dir     : {DATA_DIR.resolve()}")
print(f"Figures dir  : {FIGS_DIR.resolve()}")

## 2 — Pick best SALT model available in sncosmo

In [ ]:
SALT_SOURCE = None
for src_name in SALT_SOURCES_PRIORITY:
    try:
        _model = sncosmo.Model(source=src_name)
        SALT_SOURCE = src_name
        print(f"SALT source selected: {SALT_SOURCE}")
        break
    except Exception as e:
        print(f"  {src_name} not available: {e}")

if SALT_SOURCE is None:
    raise RuntimeError("No SALT model available. Install sncosmo and its data: sncosmo.download_builtins()")

## 3 — Download src (alert) light curve from Fink

Endpoint: `/api/v1/sources?diaObjectId=<id>`  
Returns per-visit DIA photometry: `r:psfFlux` in nJy.

In [ ]:
# Columns to fetch from the catalog endpoint
CATALOG_COLUMNS = ",".join(
    [
        "r:diaObjectId",
        "r:diaSourceId",
        "r:midpointMjdTai",
        "r:ra",
        "r:dec",
        "r:band",
        "r:visit",
        "r:detector",
        "r:x",
        "r:y",
        "r:target_name",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:scienceFlux",
        "r:scienceFluxErr",
        "r:apFlux",
        "r:apFluxErr",
        "r:templateFlux",
        "r:templateFluxErr",
        "r:snr",
        "r:reliability",
        "r:extendedness",
        "r:isNegative",
        "r:isDipole",
        "r:dipoleFluxDiff",
        "r:dipoleFluxDiffErr",
        "r:dipoleMeanFlux",
        "r:dipoleLength",
        "r:dipoleAngle",
        "r:dipoleMeanFluxErr",
        "r:dipoleNdata",
        "r:dipoleChi2",
        "r:dipoleFitAttempted",
        # Fink SN classifiers
        "f:clf_cats_class",
        "f:clf_cats_score",
        "f:clf_snnSnVsOthers_score",
        "f:clf_earlySNIa_score",
        # TNS cross-match  ← key columns for this notebook
        "f:xm_tns_fullname",
        "f:xm_tns_type",
        "f:xm_tns_redshift",  # spectroscopic z from TNS (may be NaN)
        # Other cross-matches (host galaxy proxies)
        "f:xm_simbad_otype",
        "f:xm_legacydr8_zphot",  # photometric z from Legacy Survey DR8
        "f:xm_legacydr8_e_zphot",
        "f:xm_legacydr8_fqual",
        "f:xm_legacydr8_pstar",
        "f:xm_mangrove_lum_dist",
        "f:xm_gaiadr3_DR3Name",
        "f:xm_gaiadr3_VarFlag",
        "f:xm_gaiadr3_Plx",
        "f:xm_gaiadr3_e_Plx",
        "f:xm_gcvs_type",
        "f:xm_mangrove_2MASS_name",
        "f:xm_mangrove_HyperLEDA_name",
        "f:xm_mangrove_ang_dist",
        "f:xm_mangrove_lum_dist",
        "f:xm_vsx_type",
        "f:xm_x3hsp_type",
        "f:xm_x4lac_type",
    ]
)


SRC_COLUMNS = ",".join(
    [
        "r:diaObjectId",
        "r:diaSourceId",
        "r:midpointMjdTai",
        "r:band",
        "r:ra",
        "r:dec",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:snr",
        "r:reliability",
        "r:isNegative",
        "r:isDipole",
        "r:dipoleFluxDiff",
        "r:dipoleFluxDiffErr",
        "r:dipoleMeanFlux",
        "r:dipoleLength",
        "r:dipoleAngle",
        "r:dipoleMeanFluxErr",
        "r:dipoleNdata",
        "r:dipoleChi2",
        "r:dipoleFitAttempted",
    ]
)


# ── Cross-match columns disponibles via /api/v1/sources ──────────────────────
XM_COLUMNS = ",".join(
    [
        "r:diaObjectId",
        "r:midpointMjdTai",
        # TNS
        "f:xm_tns_fullname",
        "f:xm_tns_type",
        "f:xm_tns_redshift",
        # LegacyDR8 photo-z
        "f:xm_legacydr8_zphot",
        "f:xm_legacydr8_e_zphot",
        "f:xm_legacydr8_fqual",
        # Mangrove (galaxy host)
        "f:xm_mangrove_lum_dist",
        "f:xm_mangrove_HyperLEDA_name",
        # Fink SN classifiers
        "f:clf_cats_class",
        "f:clf_snnSnVsOthers_score",
        "f:clf_earlySNIa_score",
    ]
)


def fetch_catalog(tag: str, n: int) -> pd.DataFrame:
    """Query /api/v1/tags for a given Fink LSST tag."""
    params = {
        "tag": tag,
        "n": n,
        "columns": CATALOG_COLUMNS,
        "output-format": "json",
    }
    print(f"Querying {FINK_API}/tags  tag={tag}  n={n} ...")
    resp = requests.get(f"{FINK_API}/tags", params=params, timeout=120)
    resp.raise_for_status()
    if not resp.text.strip():
        print("[warn] Empty response from API — tag may have no data yet.")
        return pd.DataFrame()
    df = pd.read_json(io.BytesIO(resp.content))
    print(f"  -> {len(df)} alerts received.")
    return df


def fetch_src(obj_id: int, cache_dir: Path, force_download: bool = False) -> pd.DataFrame:
    """Download or reload from cache the DIA alert light curve for one diaObjectId."""
    cache_path = cache_dir / f"src_{obj_id}.parquet"
    if cache_path.exists() and not force_download:
        print(f"[src] Loading from cache: {cache_path}")
        return pd.read_parquet(cache_path)

    params = {
        "diaObjectId": obj_id,
        "columns": SRC_COLUMNS,
        "output-format": "json",
    }
    print(f"[src] Querying Fink API for diaObjectId={obj_id} ...")
    resp = requests.get(f"{FINK_API}/sources", params=params, timeout=120)
    resp.raise_for_status()

    if not resp.text.strip():
        return pd.DataFrame()
    try:
        df = pd.read_json(io.BytesIO(resp.content))
        df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)
        df.to_parquet(cache_path, index=False)
        print(f"[src] {len(df)} alerts → saved to {cache_path}")
        time.sleep(API_DELAY)
    except Exception:
        return pd.DataFrame()
    if "r:midpointMjdTai" in df.columns:
        df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)
    return df


src_df = fetch_src(FINK_ID, DATA_DIR)
print(f"src shape : {src_df.shape}")
src_df.head()

## 4 — Download fp (forced photometry) light curve from Fink

Endpoint: `/api/v1/forcedphotometry?objectId=<FinkID>`  
Returns forced-position photometry at the DIA source location, even on non-detection visits.

> **Note**: the forced-photometry endpoint uses the **Fink internal ID** (`FINK_ID`),
> NOT the Rubin `diaObjectId`.

In [ ]:
FP_COLUMNS = (
    "r:diaObjectId,"
    "r:midpointMjdTai,"
    "r:psfFlux,r:psfFluxErr,r:band,"
    "r:visit,r:detector,r:x,r:y,r:xErr,r:yErr,"
    "r:scienceFlux,r:scienceFluxErr",
    "r:ra",
    "r:dec",
)


def fetch_fp(fink_id: int, cache_dir: Path, force_download: bool = False) -> pd.DataFrame:
    """Download or reload from cache the forced-photometry light curve.

    Uses Rubin diaObjectId (= Fink internal ID for LSST alerts).
    Returns columns: r:midpointMjdTai, r:band, r:psfFlux, r:psfFluxErr,
                     r:scienceFlux, r:scienceFluxErr, …
    """
    cache_path = cache_dir / f"fp_{fink_id}.parquet"
    if cache_path.exists() and not force_download:
        print(f"[fp ] Loading from cache: {cache_path}")
        return pd.read_parquet(cache_path)

    params = {
        "diaObjectId": str(fink_id),  # ← FIX: était "objectId"
        "columns": FP_COLUMNS,
        "output-format": "json",
    }
    # url = f"{FINK_API}/api/v1/fp"      # ← FIX: endpoint complet
    url = f"{FINK_API}/fp"
    print(f"[fp ] Querying Fink forced-photometry for diaObjectId={fink_id} ...")
    resp = requests.get(url, params=params, timeout=120)
    resp.raise_for_status()

    if not resp.text.strip():
        print("[fp ] Empty response — forced photometry may not be available for this object.")
        return pd.DataFrame()

    df = pd.read_json(io.BytesIO(resp.content))

    # Convert MJD column name (fp uses r:midpointMjdTai, not d:jd)
    if "r:midpointMjdTai" in df.columns:
        df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)

    df.to_parquet(cache_path, index=False)
    print(f"[fp ] {len(df)} measurements → saved to {cache_path}")
    time.sleep(API_DELAY)
    return df


# fp_df = fetch_fp(FINK_ID, DATA_DIR)
fp_df = fetch_fp(DIA_OBJECT_ID, DATA_DIR, force_download=True)
print(f"fp shape : {fp_df.shape}")
if not fp_df.empty:
    fp_df.head()

In [ ]:
def lum_dist_to_z(
    d_L_Mpc: float,
    H0: float = 70.0,
    OmegaM: float = 0.3,
    OmegaL: float = 0.7,
    z_max: float = 3.0,
    n_grid: int = 1000,
) -> float:
    """Invert the luminosity distance – redshift relation numerically."""
    c = 2.998e5  # km/s
    z_grid = np.linspace(0.001, z_max, n_grid)
    d_grid = np.array(
        [
            (c * (1 + z) / H0) * quad(lambda zp: 1.0 / np.sqrt(OmegaM * (1 + zp) ** 3 + OmegaL), 0, z)[0]
            for z in z_grid
        ]
    )
    return float(z_grid[np.argmin(np.abs(d_grid - d_L_Mpc))])

## 5 — Retrieve redshift

Priority order:
1. `f:xm_tns_redshift` — spectroscopic z from TNS
2. `f:xm_legacydr8_zphot` — photometric z from Legacy Survey DR8
3. MANGROVE luminosity distance → z converted assuming ΛCDM
4. z free (fall back to free-z fit)

In [ ]:
def fetch_redshift(dia_object_id: int) -> dict:
    """Fetch cross-match info (TNS + LegacyDR8 + Mangrove) for one diaObjectId.

    Strategy:
      1. /api/v1/sources  → grab the LAST alert row (most up-to-date xm columns)
      2. /api/v1/resolver (tns, reverse=True) → resolve diaObjectId → TNS name

    Returns a dict with keys:
        tns_name, tns_type, z_tns, z_legacydr8, ez_legacydr8,
        lum_dist_mangrove, host_name, clf_class, z_best, z_source
    """
    result = {
        "tns_name": None,
        "tns_type": None,
        "z_tns": None,
        "z_legacydr8": None,
        "ez_legacydr8": None,
        "lum_dist_mangrove": None,
        "host_name": None,
        "clf_class": None,
        "z_best": None,
        "z_source": "none",
    }

    # ── Appel 1 : une alerte pour récupérer les colonnes f: ──────────────────
    params = {
        "diaObjectId": str(dia_object_id),
        "columns": XM_COLUMNS,
        "output-format": "json",
    }
    resp = requests.get(f"{FINK_API}/sources", params=params, timeout=60)
    resp.raise_for_status()
    if resp.text.strip() and resp.text.strip() != "[]":
        df = pd.read_json(io.BytesIO(resp.content))
        if not df.empty:
            # Prendre la dernière alerte (cross-match le plus récent)
            row = df.sort_values("r:midpointMjdTai").iloc[-1]

            def _val(col):
                v = row.get(col, None)
                return None if (v is None or (isinstance(v, float) and np.isnan(v))) else v

            result["tns_name"] = _val("f:xm_tns_fullname")
            result["tns_type"] = _val("f:xm_tns_type")
            result["z_tns"] = _val("f:xm_tns_redshift")
            result["z_legacydr8"] = _val("f:xm_legacydr8_zphot")
            result["ez_legacydr8"] = _val("f:xm_legacydr8_e_zphot")
            result["lum_dist_mangrove"] = _val("f:xm_mangrove_lum_dist")
            result["host_name"] = _val("f:xm_mangrove_HyperLEDA_name")
            result["clf_class"] = _val("f:clf_cats_class")

    # ── Appel 2 : resolver TNS (reverse) pour confirmer le nom TNS ───────────
    # Utile si xm_tns_fullname est absent ou NaN dans les alertes
    if result["tns_name"] is None:
        try:
            params_r = {
                "resolver": "tns",
                "name_or_id": str(dia_object_id),
                "reverse": "True",
                "nmax": 1,
                "output-format": "json",
            }
            resp_r = requests.get(f"{FINK_API}/resolver", params=params_r, timeout=30)
            if resp_r.ok and resp_r.text.strip() not in ("[]", "{}"):
                df_r = pd.read_json(io.BytesIO(resp_r.content))
                if not df_r.empty:
                    result["tns_name"] = df_r.iloc[0].get("d:fullname", None)
                    result["tns_type"] = df_r.iloc[0].get("d:type", None)
                    # Le resolver TNS ne retourne pas le redshift directement
        except Exception:
            pass
    time.sleep(API_DELAY)

    # ── Sélection du meilleur redshift disponible ─────────────────────────────
    # Priorité : z_tns (spec) > z_legacydr8 (photo) > z_mangrove (lum_dist)
    if result["z_tns"] is not None:
        result["z_best"] = float(result["z_tns"])
        result["z_source"] = "TNS spec-z"
    elif result["z_legacydr8"] is not None:
        result["z_best"] = float(result["z_legacydr8"])
        result["z_source"] = "LegacyDR8 photo-z"
    elif result["lum_dist_mangrove"] is not None:
        # Convertir D_L (Mpc) → z approx via cosmologie plate ΛCDM
        from astropy.cosmology import FlatLambdaCDM

        cosmo = FlatLambdaCDM(H0=H0, Om0=OmegaM)
        from astropy.cosmology import z_at_value
        import astropy.units as u

        try:
            z_mangrove = float(
                z_at_value(
                    cosmo.luminosity_distance,
                    float(result["lum_dist_mangrove"]) * u.Mpc,
                    zmin=1e-4,
                    zmax=2.0,
                )
            )
            result["z_best"] = z_mangrove
            result["z_source"] = "Mangrove D_L→z"
        except Exception:
            pass

    return result

In [ ]:
# ── Appel ─────────────────────────────────────────────────────────────────────
xm = fetch_redshift(FINK_ID)

print(f"TNS name        : {xm['tns_name']}")
print(f"TNS type        : {xm['tns_type']}")
print(f"z TNS  (spec)   : {xm['z_tns']}")
print(f"z LegacyDR8     : {xm['z_legacydr8']} ± {xm['ez_legacydr8']}")
print(f"D_L Mangrove    : {xm['lum_dist_mangrove']} Mpc  (host: {xm['host_name']})")
print(f"Fink class      : {xm['clf_class']}")
print(f"─────────────────────────────────────────")
print(f"z_best          : {xm['z_best']}  [{xm['z_source']}]")

# Variables pour la suite du notebook
tns_name = xm["tns_name"]
tns_type = xm["tns_type"]
z_used = xm["z_best"]
z_source = xm["z_source"]

if z_used is None:
    print("\n[warn] No redshift found. Will run free-z fit only.")

## 6 — Inspect light curves (src + fp)

In [ ]:
# ── Band statistics for src ───────────────────────────────────────────────────
if not src_df.empty and "r:band" in src_df.columns:
    print("src — alerts per band:")
    print(src_df["r:band"].value_counts().to_string())
    print(f"MJD range: {src_df['r:midpointMjdTai'].min():.2f} – {src_df['r:midpointMjdTai'].max():.2f}")
    mjd_start = src_df["r:midpointMjdTai"].min()
    t_start = Time(mjd_start, format="mjd")
    date_start_str = t_start.to_value("iso", subfmt="date")  # YYYY-MM-DD
    print(f"MJD start: {mjd_start:.2f} – {date_start_str}")


if not fp_df.empty and "d:band" in fp_df.columns:
    print("\nfp — measurements per band:")
    print(fp_df["d:band"].value_counts().to_string())
    print(f"MJD range: {fp_df['mjd'].min():.2f} – {fp_df['mjd'].max():.2f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)
title_suffix = f"diaObjectId={DIA_OBJECT_ID}"
if tns_name:
    title_suffix += f" / {tns_name}"
if z_used is not None:
    title_suffix += f" / z={z_used:.4f} ({z_source})"

# ── Panel 1: src (DIA psfFlux) ────────────────────────────────────────────────
ax = axes[0]
if not src_df.empty:
    for band in BAND_ORDER:
        mask = src_df["r:band"] == band
        if mask.sum() == 0:
            continue
        sub = src_df[mask]
        snr = sub["r:psfFlux"] / sub["r:psfFluxErr"]
        ax.errorbar(
            sub["r:midpointMjdTai"],
            sub["r:psfFlux"],
            yerr=sub["r:psfFluxErr"],
            fmt="o",
            color=BAND_COLORS.get(band, "grey"),
            label=f"{band} (N={mask.sum()}, SNR={np.sqrt((snr**2).sum()):.1f})",
            markersize=4,
            linewidth=0.8,
            capsize=2,
        )
    ax.axhline(0, color="grey", lw=0.5, ls="--")
    ax.set_xlabel("MJD")
    ax.set_ylabel("psfFlux (nJy)")
    ax.set_title(f"src (DIA alerts) — {title_suffix} start date : {date_start_str}")
    ax.legend(fontsize=8, ncol=3)
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No src data", ha="center", va="center", transform=ax.transAxes)

# ── Panel 2: fp (forced photometry) ──────────────────────────────────────────
ax = axes[1]
if not fp_df.empty:
    for band in BAND_ORDER:
        mask = fp_df["r:band"] == band
        if mask.sum() == 0:
            continue
        sub = fp_df[mask]
        ax.errorbar(
            sub["r:midpointMjdTai"],
            sub["r:psfFlux"],
            yerr=sub["r:psfFluxErr"],
            fmt="o",
            markerfacecolor="none",  # ← marker vide
            markeredgecolor=BAND_COLORS.get(band, "grey"),  # ← contour coloré
            markeredgewidth=1.2,  # optionnel mais recommandé
            ecolor=BAND_COLORS.get(band, "grey"),  # couleur des barres d'erreur
            linestyle="none",  # ← évite de relier les points
            label=f"{band} (N={mask.sum()})",
            markersize=4,
            linewidth=0.3,
            capsize=2,
        )
    ax.axhline(0, color="grey", lw=0.5, ls="--")
    ax.set_xlabel("MJD")
    ax.set_ylabel("fpFlux (nJy)")
    ax.set_title("fp (forced photometry)")
    ax.legend(fontsize=8, ncol=3)
    ax.grid(True, alpha=0.3)
else:
    ax.text(
        0.5,
        0.5,
        "No forced-photometry data available",
        ha="center",
        va="center",
        transform=ax.transAxes,
        color="grey",
    )

plt.tight_layout()
fig.savefig(FIGS_DIR / "lightcurves_src_fp_separated.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 6), layout="constrained")
title_suffix = f"diaObjectId={DIA_OBJECT_ID}"
if tns_name:
    title_suffix += f" / {tns_name}"
if z_used is not None:
    title_suffix += f" / z={z_used:.4f} ({z_source})"

# ── Panel 1: src (DIA psfFlux) ────────────────────────────────────────────────
ax = axes
if not src_df.empty:
    for band in BAND_ORDER:
        mask = src_df["r:band"] == band
        if mask.sum() == 0:
            continue
        sub = src_df[mask]
        snr = sub["r:psfFlux"] / sub["r:psfFluxErr"]
        ax.errorbar(
            sub["r:midpointMjdTai"],
            sub["r:psfFlux"],
            yerr=sub["r:psfFluxErr"],
            fmt="o",
            color=BAND_COLORS.get(band, "grey"),
            label=f"{band} (N={mask.sum()}, SNR={np.sqrt((snr**2).sum()):.1f})",
            markersize=4,
            linewidth=0.8,
            capsize=2,
        )
    ax.axhline(0, color="grey", lw=0.5, ls="--")
    ax.set_xlabel("MJD")
    ax.set_ylabel("psfFlux (nJy)")
    ax.set_title(f"src (DIA alerts) — {title_suffix} start date : {date_start_str}")
    ax.legend(fontsize=8, ncol=3)
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No src data", ha="center", va="center", transform=ax.transAxes)

# ── Panel 2: fp (forced photometry) ──────────────────────────────────────────
ax = axes
if not fp_df.empty:
    for band in BAND_ORDER:
        mask = fp_df["r:band"] == band
        if mask.sum() == 0:
            continue
        sub = fp_df[mask]
        ax.errorbar(
            sub["r:midpointMjdTai"],
            sub["r:psfFlux"],
            yerr=sub["r:psfFluxErr"],
            fmt="o",
            markerfacecolor="none",  # ← marker vide
            markeredgecolor=BAND_COLORS.get(band, "grey"),  # ← contour coloré
            markeredgewidth=1.2,  # optionnel mais recommandé
            ecolor=BAND_COLORS.get(band, "grey"),  # couleur des barres d'erreur
            linestyle="none",  # ← évite de relier les points
            label=f"{band} (N={mask.sum()})",
            markersize=4,
            linewidth=0.3,
            capsize=2,
        )
    # ax.axhline(0, color="grey", lw=0.5, ls="--")
    # ax.set_xlabel("MJD")
    # ax.set_ylabel("fpFlux (nJy)")
    # ax.set_title("fp (forced photometry)")
    ax.legend(fontsize=8, ncol=3)
    # ax.grid(True, alpha=0.3)
else:
    ax.text(
        0.5,
        0.5,
        "No forced-photometry data available",
        ha="center",
        va="center",
        transform=ax.transAxes,
        color="grey",
    )

# plt.tight_layout()
fig.savefig(FIGS_DIR / "lightcurves_src_fp_combined.png", dpi=150, bbox_inches="tight")
plt.show()

## 7 — Build sncosmo observation table from src

Quality cuts:
- SNR = `psfFlux / psfFluxErr` ≥ `SNR_MIN`
- Keep only positive flux points (non-negative detections)
- Band must be in LSST band set {u, g, r, i, z, y}

In [ ]:
def build_sncosmo_table(
    df: pd.DataFrame,
    mjd_col: str = "r:midpointMjdTai",
    flux_col: str = "r:psfFlux",
    fluxerr_col: str = "r:psfFluxErr",
    band_col: str = "r:band",
    snr_min: float = SNR_MIN,
    band_prefix: str = BAND_PREFIX,
    zp: float = RUBIN_ZP,
    zpsys: str = ZPSYS,
) -> astropy.table.Table:
    """Convert a Fink src DataFrame to an sncosmo-compatible observation Table.

    Returns an astropy Table with columns: time, band, flux, fluxerr, zp, zpsys.
    """
    if df.empty:
        raise ValueError("Input DataFrame is empty.")

    df = df.copy()
    df["snr_"] = df[flux_col] / df[fluxerr_col]

    # Quality cuts
    mask = df[fluxerr_col] > 0 & df[flux_col].notna() & df[fluxerr_col].notna() & (
        df["snr_"] >= snr_min
    ) & df[band_col].isin(BAND_ORDER)
    obs = df[mask].copy()

    print(f"Quality cuts: {len(df)} → {len(obs)} points (SNR ≥ {snr_min})")
    if len(obs) == 0:
        raise ValueError("No points survive quality cuts.")

    table = astropy.table.Table(
        {
            "time": obs[mjd_col].values.astype(float),
            "band": [f"{band_prefix}{b}" for b in obs[band_col].values],
            "flux": obs[flux_col].values.astype(float),
            "fluxerr": obs[fluxerr_col].values.astype(float),
            "zp": np.full(len(obs), zp),
            "zpsys": [zpsys] * len(obs),
        }
    )
    return table


obs_table = build_sncosmo_table(src_df)
print(f"sncosmo table: {len(obs_table)} points across bands {set(obs_table['band'])}")
obs_table[:5]

## 8 — SALT3 fit with fixed redshift

Only run if a redshift was found in section 5.

In [ ]:
def plot_fisher_ellipses(
    param_names,
    theta_hat,
    cov,
    levels=(2.30, 6.17),  # 68%, 95%
    figsize=(6, 6),
):
    """
    Plot Fisher correlation ellipses from covariance matrix.

    Parameters
    ----------
    param_names : list of str
    theta_hat   : array-like (N,)
    cov         : array-like (N,N)
    levels      : tuple of Δχ² values
    """

    n = len(param_names)
    fig, axes = plt.subplots(n, n, figsize=(2 * n, 2 * n))

    # loop on parameters
    for i in range(n):
        for j in range(n):
            ax = axes[i, j]

            # auto correlation
            if i == j:
                # 1D Gaussian
                sigma = np.sqrt(cov[i, i])
                x = np.linspace(theta_hat[i] - 4 * sigma, theta_hat[i] + 4 * sigma, 200)
                y = np.exp(-((x - theta_hat[i]) ** 2) / (2 * sigma**2))
                ax.plot(x, y)
                ax.set_yticks([])

            # ellipses : 2 parameter correlation
            elif i > j:
                # 2D ellipse
                # sub convariance 2x2
                sub_cov = cov[np.ix_([j, i], [j, i])]
                # diagonalisation
                vals, vecs = np.linalg.eigh(sub_cov)

                # tri des valeurs propres
                order = vals.argsort()[::-1]
                vals = vals[order]
                vecs = vecs[:, order]

                angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))

                for level in levels:
                    width = 2 * np.sqrt(vals[0] * level)
                    height = 2 * np.sqrt(vals[1] * level)

                    ellipse = Ellipse(
                        xy=(theta_hat[j], theta_hat[i]), width=width, height=height, angle=angle, fill=False
                    )
                    ax.add_patch(ellipse)

                ax.scatter(theta_hat[j], theta_hat[i], s=10)

            else:
                ax.axis("off")

            if i == n - 1:
                ax.set_xlabel(param_names[j])
            if j == 0:
                ax.set_ylabel(param_names[i])

    plt.tight_layout()
    return fig

In [ ]:
fit_fixed_z = None
fit_fixed_z_cov = None

if z_used is not None and len(obs_table) >= MIN_DATAPOINTS:
    model = sncosmo.Model(source=SALT_SOURCE)
    model.set(z=z_used)

    # Estimate t0 from peak flux
    t0_init = float(obs_table["time"][np.argmax(obs_table["flux"])])
    model.set(t0=t0_init)

    print(f"Fitting {SALT_SOURCE} with z fixed = {z_used:.5f}  (source: {z_source})")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            res, fitted_model = sncosmo.fit_lc(
                obs_table,
                model,
                ["t0", "x0", "x1", "c"],
                bounds={"x1": (-5, 5), "c": (-1, 2)},
            )
            fit_fixed_z = res
            fit_fixed_z_cov = res.covariance
            print(f"  status  : {res.message}")
            print(f"  t0      : {res.parameters[res.param_names.index('t0')]:.3f}")
            print(f"  x0      : {res.parameters[res.param_names.index('x0')]:.4e}")
            print(f"  x1      : {res.parameters[res.param_names.index('x1')]:.4f}")
            print(f"  c       : {res.parameters[res.param_names.index('c')]:.4f}")
            print(f"  χ²/dof  : {res.chisq:.2f} / {res.ndof}")
        except Exception as exc:
            print(f"  [error] fit failed: {exc}")
else:
    print("Skipping fixed-z fit (no redshift or insufficient data).")

In [ ]:
# ── Plot SALT3 fit (fixed z) ──────────────────────────────────────────────────
if fit_fixed_z is not None:
    fig = sncosmo.plot_lc(
        obs_table,
        model=fitted_model,
        errors=res.errors,
        figtext=(
            f"{tns_name or 'diaObj ' + str(DIA_OBJECT_ID)}  "
            f"z={z_used:.4f} ({z_source})\n"
            f"x1={res.parameters[res.param_names.index('x1')]:.3f}  "
            f"c={res.parameters[res.param_names.index('c')]:.3f}  "
            f"χ²/dof={res.chisq:.1f}/{res.ndof}"
        ),
    )
    fig.savefig(FIGS_DIR / "salt3_fit_fixed_z.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved → {FIGS_DIR}/salt3_fit_fixed_z.png")

## 9 — SALT3 fit with free redshift (two-pass)

**Pass 1** — coarse grid scan: fix z on `N_GRID_Z` values, fit `(t0, x0, x1, c)`,
record χ².  
**Pass 2** — refined free-z fit initialised at the grid minimum.

In [ ]:
fit_free_z = None
fit_free_z_model = None
z_grid_best = None
chi2_grid = None
z_grid_vals = None

if len(obs_table) >= MIN_DATAPOINTS:
    print("Pass 1 — coarse redshift grid scan ...")
    z_grid_vals = np.linspace(Z_BOUND_LO, Z_BOUND_HI, N_GRID_Z)
    chi2_grid = np.full(N_GRID_Z, np.inf)
    t0_init = float(obs_table["time"][np.argmax(obs_table["flux"])])

    for i, z_try in enumerate(z_grid_vals):
        m = sncosmo.Model(source=SALT_SOURCE)
        m.set(z=z_try, t0=t0_init)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                r, _ = sncosmo.fit_lc(
                    obs_table,
                    m,
                    ["t0", "x0", "x1", "c"],
                    bounds={"x1": (-5, 5), "c": (-1, 2)},
                )
                if r.success:
                    chi2_grid[i] = r.chisq
            except Exception:
                pass

    best_idx = int(np.argmin(chi2_grid))
    z_grid_best = z_grid_vals[best_idx]
    print(f"  Grid best z={z_grid_best:.4f}  χ²={chi2_grid[best_idx]:.2f}")

    print("Pass 2 — refined free-z fit ...")
    model2 = sncosmo.Model(source=SALT_SOURCE)
    # Re-run grid best to get good initial t0, x0, x1, c
    model2.set(z=z_grid_best, t0=t0_init)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            r_init, _ = sncosmo.fit_lc(
                obs_table,
                model2,
                ["t0", "x0", "x1", "c"],
                bounds={"x1": (-5, 5), "c": (-1, 2)},
            )
            model2.set(**{p: r_init.parameters[r_init.param_names.index(p)] for p in ["t0", "x0", "x1", "c"]})
        except Exception:
            pass

    z_lo = max(Z_BOUND_LO, z_grid_best - Z_REFINE_HALF)
    z_hi = min(Z_BOUND_HI, z_grid_best + Z_REFINE_HALF)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            res2, fitted_model2 = sncosmo.fit_lc(
                obs_table,
                model2,
                ["z", "t0", "x0", "x1", "c"],
                bounds={"z": (z_lo, z_hi), "x1": (-5, 5), "c": (-1, 2)},
            )
            fit_free_z = res2
            fit_free_z_model = fitted_model2
            z_fit = res2.parameters[res2.param_names.index("z")]
            x1_fit = res2.parameters[res2.param_names.index("x1")]
            c_fit = res2.parameters[res2.param_names.index("c")]
            print(f"  Free-z result:")
            print(f"    z_fit  = {z_fit:.5f}  (σ_z = {res2.errors.get('z', np.nan):.5f})")
            print(f"    x1     = {x1_fit:.4f}")
            print(f"    c      = {c_fit:.4f}")
            print(f"    χ²/dof = {res2.chisq:.2f}/{res2.ndof}")
            if z_used is not None:
                dz = z_fit - z_used
                print(f"    Δz(fit-ref) = {dz:+.5f}  |  Δz/(1+z_ref) = {dz / (1 + z_used):+.5f}")
        except Exception as exc:
            print(f"  [error] free-z fit failed: {exc}")
else:
    print("Skipping free-z fit (insufficient data points).")

In [ ]:
# ── Plot χ² grid ─────────────────────────────────────────────────────────────
if z_grid_vals is not None and chi2_grid is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    finite = np.isfinite(chi2_grid)
    ax.plot(z_grid_vals[finite], chi2_grid[finite], "b-", lw=1.5, label="χ² grid")
    ax.axvline(z_grid_best, color="red", ls="--", label=f"z_grid_best={z_grid_best:.4f}")
    if fit_free_z is not None:
        z_fit = fit_free_z.parameters[fit_free_z.param_names.index("z")]
        ax.axvline(z_fit, color="orange", ls=":", lw=2, label=f"z_fit_free={z_fit:.4f}")
    if z_used is not None:
        ax.axvline(z_used, color="green", ls="-.", lw=2, label=f"z_ref={z_used:.4f} ({z_source})")
    ax.set_xlabel("Redshift z")
    ax.set_ylabel("χ²")
    ax.set_title(f"SALT3 — χ² grid scan  (diaObjectId={DIA_OBJECT_ID})")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIGS_DIR / "salt3_chi2_grid.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Plot SALT3 fit (free z) ───────────────────────────────────────────────────
if fit_free_z is not None:
    z_fit = fit_free_z.parameters[fit_free_z.param_names.index("z")]
    x1_fit = fit_free_z.parameters[fit_free_z.param_names.index("x1")]
    c_fit = fit_free_z.parameters[fit_free_z.param_names.index("c")]
    sz = fit_free_z.errors.get("z", np.nan)
    fig = sncosmo.plot_lc(
        obs_table,
        model=fit_free_z_model,
        errors=fit_free_z.errors,
        figtext=(
            f"{tns_name or 'diaObj ' + str(DIA_OBJECT_ID)}\n"
            f"z_fit={z_fit:.4f}±{sz:.4f}  "
            + (f"z_ref={z_used:.4f} ({z_source})\n" if z_used is not None else "\n")
            + f"x1={x1_fit:.3f}  c={c_fit:.3f}  χ²/dof={fit_free_z.chisq:.1f}/{fit_free_z.ndof}"
        ),
    )
    fig.savefig(FIGS_DIR / "salt3_fit_free_z.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved → {FIGS_DIR}/salt3_fit_free_z.png")

## 10 — Ellipses de corrélation Fisher (SALT3)

On extrait la matrice de covariance retournée par `sncosmo.fit_lc` et on trace
les ellipses de corrélation à 68 % et 95 % de niveau de confiance (Δχ² = 2.30 et 6.17
pour 2 degrés de liberté).

**Paramètres affichés** : `t0`, `x0`, `x1`, `c` (fit z fixé) et `z`, `t0`, `x0`, `x1`, `c` (fit z libre).  
La diagonale montre la gaussienne marginale 1D ; le triangle inférieur montre les ellipses 2D.

In [ ]:
# ── Imports for Fisher ellipses ──────────────────────────────────────────────
from matplotlib.patches import Ellipse
from scipy.stats import chi2 as scipy_chi2

ndeg = 2  # 2 d.o.f. for 2D confidence ellipses
DELTA_CHI2_68 = scipy_chi2.ppf(0.683, df=ndeg)  # ≈ 2.30
DELTA_CHI2_95 = scipy_chi2.ppf(0.954, df=ndeg)  # ≈ 6.17
print(f"Δχ² 68% = {DELTA_CHI2_68:.4f}   Δχ² 95% = {DELTA_CHI2_95:.4f}")

In [ ]:
def plot_fisher_ellipses(param_names, theta_hat, cov, levels=(2.30, 6.17), figsize=None, title=None):
    """
    Draw Fisher correlation ellipses from a covariance matrix.

    Parameters
    ----------
    param_names : list[str]         parameter names (N)
    theta_hat   : array-like (N,)   best-fit values
    cov         : array-like (N,N)  covariance matrix
    levels      : tuple of Delta_chi2  (68 %, 95 %) -> (2.30, 6.17)
    figsize     : tuple or None
    title       : str or None

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    n = len(param_names)
    if figsize is None:
        figsize = (2.5 * n, 2.5 * n)
    cov = np.asarray(cov)
    theta_hat = np.asarray(theta_hat)
    fig, axes = plt.subplots(n, n, figsize=figsize)
    if n == 1:
        axes = np.array([[axes]])
    colors = ["C0", "C1"]
    labels = ["68 % CL", "95 % CL"]
    for i in range(n):
        for j in range(n):
            ax = axes[i, j]
            if i == j:
                # Diagonal: 1D marginal Gaussian
                sigma = np.sqrt(cov[i, i])
                x = np.linspace(theta_hat[i] - 4 * sigma, theta_hat[i] + 4 * sigma, 300)
                y = np.exp(-((x - theta_hat[i]) ** 2) / (2 * sigma**2))
                ax.plot(x, y, color="C0")
                ax.axvline(theta_hat[i], color="k", lw=0.8, ls="--")
                ax.fill_between(x, y, where=(np.abs(x - theta_hat[i]) <= sigma), alpha=0.25, color="C0")
                ax.set_yticks([])
                ax.set_ylim(0, 1.15)
            elif i > j:
                # Lower triangle: 2D Fisher ellipses
                sub_cov = cov[np.ix_([j, i], [j, i])]
                vals, vecs = np.linalg.eigh(sub_cov)
                order = vals.argsort()[::-1]
                vals, vecs = vals[order], vecs[:, order]
                angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
                r_max = 3 * np.sqrt(vals[0] * levels[-1])
                ax.set_xlim(theta_hat[j] - r_max, theta_hat[j] + r_max)
                ax.set_ylim(theta_hat[i] - r_max, theta_hat[i] + r_max)
                for k, (level, color, label) in enumerate(zip(levels, colors, labels)):
                    width = 2 * np.sqrt(vals[0] * level)
                    height = 2 * np.sqrt(vals[1] * level)
                    ell = Ellipse(
                        xy=(theta_hat[j], theta_hat[i]),
                        width=width,
                        height=height,
                        angle=angle,
                        fill=(k == 0),
                        facecolor=color if k == 0 else "none",
                        alpha=0.20 if k == 0 else 1.0,
                        edgecolor=color,
                        linewidth=1.5,
                        label=label if (j == 0 and i == 1) else None,
                    )
                    ax.add_patch(ell)
                ax.scatter(theta_hat[j], theta_hat[i], s=25, color="k", zorder=5)
                rho = sub_cov[0, 1] / np.sqrt(sub_cov[0, 0] * sub_cov[1, 1])
                ax.text(0.05, 0.93, f"ρ={rho:.3f}", transform=ax.transAxes, fontsize=7, va="top")
            else:
                # Upper triangle: blank
                ax.set_visible(False)
            if j == 0 and i > 0:
                ax.set_ylabel(param_names[i], fontsize=8)
            else:
                ax.set_ylabel("")
            if i == n - 1:
                ax.set_xlabel(param_names[j], fontsize=8)
            else:
                ax.set_xlabel("")
            ax.tick_params(labelsize=6)
    if n > 1:
        axes[1, 0].legend(fontsize=7, loc="lower right")
    if title:
        fig.suptitle(title, fontsize=9, y=1.01)
    fig.tight_layout()
    return fig


print("plot_fisher_ellipses() ready.")

### 10.1 — Ellipses Fisher — fit z fixé  (`t0, x0, x1, c`)

In [ ]:
# ── Extract covariance from fixed-z fit ─────────────────────────────────────
cov_fixed = None
if fit_fixed_z is not None and fit_fixed_z.covariance is not None:
    vpnames_fixed = list(fit_fixed_z.vparam_names)  # ['t0','x0','x1','c']
    theta_fixed = np.array([fit_fixed_z.parameters[fit_fixed_z.param_names.index(p)] for p in vpnames_fixed])
    cov_fixed = np.array(fit_fixed_z.covariance)

    print("Fixed-z best-fit parameters:")
    for name, val, sig in zip(vpnames_fixed, theta_fixed, np.sqrt(np.diag(cov_fixed))):
        print(f"  {name:>4s} = {val:.6g} ± {sig:.4g}")
    print()
    print("Covariance matrix:")
    print(cov_fixed)
else:
    print("fit_fixed_z not available or covariance is None.")

In [ ]:
# ── Figure 10.1a : all 4 parameters (t0, x0, x1, c) ────────────────────────
if cov_fixed is not None:
    fig_ell_fixed4 = plot_fisher_ellipses(
        param_names=vpnames_fixed,
        theta_hat=theta_fixed,
        cov=cov_fixed,
        levels=(DELTA_CHI2_68, DELTA_CHI2_95),
        title=(
            f"SALT3 fixed-z — diaObjectId {DIA_OBJECT_ID}  z={z_used:.4f} ({z_source})\n"
            f"Fisher ellipses 68 % & 95 % CL   \u03c7\u00b2/dof={fit_fixed_z.chisq:.1f}/{fit_fixed_z.ndof}"
        ),
    )
    fig_ell_fixed4.savefig(FIGS_DIR / "salt3_fisher_fixed_z_4params.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved \u2192 {FIGS_DIR}/salt3_fisher_fixed_z_4params.png")
else:
    print("Covariance not available for fixed-z fit.")

In [ ]:
# ── Figure 10.1b : cosmological sub-matrix (x1, c) only ────────────────────
if cov_fixed is not None:
    idx_x1 = vpnames_fixed.index("x1")
    idx_c = vpnames_fixed.index("c")
    idxs = [idx_x1, idx_c]

    theta_x1c = theta_fixed[idxs]
    cov_x1c = cov_fixed[np.ix_(idxs, idxs)]
    rho_x1c = cov_x1c[0, 1] / np.sqrt(cov_x1c[0, 0] * cov_x1c[1, 1])

    print(f"x1 = {theta_x1c[0]:.4f} \u00b1 {np.sqrt(cov_x1c[0, 0]):.4f}")
    print(f"c  = {theta_x1c[1]:.4f} \u00b1 {np.sqrt(cov_x1c[1, 1]):.4f}")
    print(f"Pearson \u03c1(x1, c) = {rho_x1c:.4f}")

    fig_x1c_fixed = plot_fisher_ellipses(
        param_names=["x1 (stretch)", "c (colour)"],
        theta_hat=theta_x1c,
        cov=cov_x1c,
        levels=(DELTA_CHI2_68, DELTA_CHI2_95),
        figsize=(5, 5),
        title=(
            f"SALT3 fixed-z — x1 vs c   \u03c1={rho_x1c:.3f}\n"
            f"z={z_used:.4f} ({z_source})   \u03c7\u00b2/dof={fit_fixed_z.chisq:.1f}/{fit_fixed_z.ndof}"
        ),
    )
    fig_x1c_fixed.savefig(FIGS_DIR / "salt3_fisher_fixed_z_x1c.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved \u2192 {FIGS_DIR}/salt3_fisher_fixed_z_x1c.png")

In [ ]:
# ── Figure 10.1b : cosmological sub-matrix (x1, c) only ────────────────────
if cov_fixed is not None:
    idx_x1 = vpnames_fixed.index("x1")
    idx_c = vpnames_fixed.index("c")
    idxs = [idx_x1, idx_c]

    theta_x1c = theta_fixed[idxs]
    cov_x1c = cov_fixed[np.ix_(idxs, idxs)]
    rho_x1c = cov_x1c[0, 1] / np.sqrt(cov_x1c[0, 0] * cov_x1c[1, 1])

    print(f"x1 = {theta_x1c[0]:.4f} \u00b1 {np.sqrt(cov_x1c[0, 0]):.4f}")
    print(f"c  = {theta_x1c[1]:.4f} \u00b1 {np.sqrt(cov_x1c[1, 1]):.4f}")
    print(f"Pearson \u03c1(x1, c) = {rho_x1c:.4f}")

    fig_x1c_fixed = plot_fisher_ellipses(
        param_names=["x1 (stretch)", "c (colour)"],
        theta_hat=theta_x1c,
        cov=cov_x1c,
        levels=(DELTA_CHI2_68, DELTA_CHI2_95),
        figsize=(5, 5),
        title=(
            f"SALT3 fixed-z — x1 vs c   \u03c1={rho_x1c:.3f}\n"
            f"z={z_used:.4f} ({z_source})   \u03c7\u00b2/dof={fit_fixed_z.chisq:.1f}/{fit_fixed_z.ndof}"
        ),
    )
    fig_x1c_fixed.savefig(FIGS_DIR / "salt3_fisher_fixed_z_x1c.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved \u2192 {FIGS_DIR}/salt3_fisher_fixed_z_x1c.png")

### 10.2 — Ellipses Fisher — fit z libre  (`z, t0, x0, x1, c`)

In [ ]:
# ── Extract covariance from free-z fit ──────────────────────────────────────
cov_free = None
if fit_free_z is not None and fit_free_z.covariance is not None:
    vpnames_free = list(fit_free_z.vparam_names)  # ['z','t0','x0','x1','c']
    theta_free = np.array([fit_free_z.parameters[fit_free_z.param_names.index(p)] for p in vpnames_free])
    cov_free = np.array(fit_free_z.covariance)

    print("Free-z best-fit parameters:")
    for name, val, sig in zip(vpnames_free, theta_free, np.sqrt(np.diag(cov_free))):
        print(f"  {name:>4s} = {val:.6g} \u00b1 {sig:.4g}")
    print()
    print("Covariance matrix:")
    print(cov_free)
else:
    print("fit_free_z not available or covariance is None.")

In [ ]:
# ── Figure 10.2a : all 5 parameters (z, t0, x0, x1, c) ─────────────────────
if cov_free is not None:
    fig_ell_free5 = plot_fisher_ellipses(
        param_names=vpnames_free,
        theta_hat=theta_free,
        cov=cov_free,
        levels=(DELTA_CHI2_68, DELTA_CHI2_95),
        title=(
            f"SALT3 free-z — diaObjectId {DIA_OBJECT_ID}\n"
            f"Fisher ellipses 68 % & 95 % CL   \u03c7\u00b2/dof={fit_free_z.chisq:.1f}/{fit_free_z.ndof}"
        ),
    )
    fig_ell_free5.savefig(FIGS_DIR / "salt3_fisher_free_z_5params.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved \u2192 {FIGS_DIR}/salt3_fisher_free_z_5params.png")
else:
    print("Covariance not available for free-z fit.")

In [ ]:
# ── Figure 10.2b : cosmological sub-matrix (x1, c) — free-z ────────────────
if cov_free is not None:
    idx_x1f = vpnames_free.index("x1")
    idx_cf = vpnames_free.index("c")
    idxsf = [idx_x1f, idx_cf]

    theta_x1c_f = theta_free[idxsf]
    cov_x1c_f = cov_free[np.ix_(idxsf, idxsf)]
    rho_x1c_f = cov_x1c_f[0, 1] / np.sqrt(cov_x1c_f[0, 0] * cov_x1c_f[1, 1])

    print(f"x1 = {theta_x1c_f[0]:.4f} \u00b1 {np.sqrt(cov_x1c_f[0, 0]):.4f}")
    print(f"c  = {theta_x1c_f[1]:.4f} \u00b1 {np.sqrt(cov_x1c_f[1, 1]):.4f}")
    print(f"Pearson \u03c1(x1, c) = {rho_x1c_f:.4f}")

    z_fit_val = fit_free_z.parameters[fit_free_z.param_names.index("z")]
    fig_x1c_free = plot_fisher_ellipses(
        param_names=["x1 (stretch)", "c (colour)"],
        theta_hat=theta_x1c_f,
        cov=cov_x1c_f,
        levels=(DELTA_CHI2_68, DELTA_CHI2_95),
        figsize=(5, 5),
        title=(
            f"SALT3 free-z — x1 vs c   \u03c1={rho_x1c_f:.3f}\n"
            f"z_fit={z_fit_val:.4f}   \u03c7\u00b2/dof={fit_free_z.chisq:.1f}/{fit_free_z.ndof}"
        ),
    )
    fig_x1c_free.savefig(FIGS_DIR / "salt3_fisher_free_z_x1c.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved \u2192 {FIGS_DIR}/salt3_fisher_free_z_x1c.png")

In [ ]:
# ── Figure 10.2c : (z, x1) — redshift vs stretch degeneracy ─────────────────
if cov_free is not None and "z" in vpnames_free:
    idx_z_f = vpnames_free.index("z")
    idx_x1_f = vpnames_free.index("x1")
    idxs_zx1 = [idx_z_f, idx_x1_f]

    theta_zx1 = theta_free[idxs_zx1]
    cov_zx1 = cov_free[np.ix_(idxs_zx1, idxs_zx1)]
    rho_zx1 = cov_zx1[0, 1] / np.sqrt(cov_zx1[0, 0] * cov_zx1[1, 1])

    print(f"Pearson \u03c1(z, x1) = {rho_zx1:.4f}")

    fig_zx1 = plot_fisher_ellipses(
        param_names=["z (redshift)", "x1 (stretch)"],
        theta_hat=theta_zx1,
        cov=cov_zx1,
        levels=(DELTA_CHI2_68, DELTA_CHI2_95),
        figsize=(5, 5),
        title=(
            f"SALT3 free-z — z vs x1   \u03c1={rho_zx1:.3f}\n"
            f"\u03c7\u00b2/dof={fit_free_z.chisq:.1f}/{fit_free_z.ndof}"
        ),
    )
    # Overlay reference redshift if available
    if z_used is not None:
        fig_zx1.axes[2].axvline(
            z_used, color="green", ls="-.", lw=1.5, label=f"z_ref={z_used:.4f} ({z_source})"
        )
        fig_zx1.axes[2].legend(fontsize=7)
    fig_zx1.savefig(FIGS_DIR / "salt3_fisher_free_z_z_x1.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved \u2192 {FIGS_DIR}/salt3_fisher_free_z_z_x1.png")

## 11 — Distance modulus (Tripp formula)

$$\mu = m_B - M_B + \alpha\,x_1 - \beta\,c$$

Error propagation (gradient on covariance matrix):
$$\sigma_\mu^2 = g^T C g, \quad g = \frac{\partial\mu}{\partial(t_0, x_0, x_1, c)}$$

Compare to flat ΛCDM: $\mu_{\rm ΛCDM}(z) = 5\log_{10}[d_L(z)/{\rm Mpc}] + 25$.

In [ ]:
def compute_mu(res, alpha=ALPHA, beta=BETA, M_B=M_B):
    """Compute distance modulus and its error from a sncosmo fit result.

    μ = -2.5*log10(x0) + 10.095 - M_B + α*x1 - β*c
    (the 10.095 comes from the SALT2/3 convention for m_B)

    Returns (mu, sigma_mu).
    """
    pnames = list(res.param_names)
    params = res.parameters
    x0 = params[pnames.index("x0")]
    x1 = params[pnames.index("x1")]
    c = params[pnames.index("c")]

    m_B_obs = -2.5 * np.log10(x0) + 10.095  # SALT2/3 peak apparent magnitude
    mu = m_B_obs - M_B + alpha * x1 - beta * c

    # Gradient wrt (x0, x1, c)
    dm_dx0 = -2.5 / (x0 * np.log(10))
    dm_dx1 = alpha
    dm_dc = -beta

    # Build gradient vector in the order of res.param_names
    # (covariance matrix has same ordering)
    n = len(pnames)
    g = np.zeros(n)
    for name, grad in [("x0", dm_dx0), ("x1", dm_dx1), ("c", dm_dc)]:
        if name in pnames:
            g[pnames.index(name)] = grad

    cov = res.covariance
    var = float(g @ cov @ g) if (cov is not None and cov.shape == (n, n)) else np.nan
    sigma_mu = np.sqrt(var) if var >= 0 else np.nan

    return mu, sigma_mu


def mu_lcdm(z, H0=H0, OmegaM=OmegaM, OmegaL=OmegaL):
    """Distance modulus in flat ΛCDM."""
    c = 2.998e5  # km/s
    if z <= 0:
        return np.nan
    integral, _ = quad(lambda zp: 1.0 / np.sqrt(OmegaM * (1 + zp) ** 3 + OmegaL), 0, z)
    d_L = (c * (1 + z) / H0) * integral  # Mpc
    return 5.0 * np.log10(d_L) + 25.0


# Compute μ for both fits
results_summary = []

if fit_fixed_z is not None:
    mu_fix, smu_fix = compute_mu(fit_fixed_z)
    mu_ref = mu_lcdm(z_used)
    print(f"Fixed-z fit:")
    print(f"  z             = {z_used:.5f}  ({z_source})")
    print(f"  μ_obs         = {mu_fix:.4f} ± {smu_fix:.4f}")
    print(f"  μ_ΛCDM(z_ref) = {mu_ref:.4f}")
    print(f"  Δμ            = {mu_fix - mu_ref:+.4f}")
    results_summary.append(
        {
            "fit": "fixed_z",
            "z": z_used,
            "z_source": z_source,
            "x1": fit_fixed_z.parameters[fit_fixed_z.param_names.index("x1")],
            "c": fit_fixed_z.parameters[fit_fixed_z.param_names.index("c")],
            "mu": mu_fix,
            "sigma_mu": smu_fix,
            "mu_lcdm": mu_ref,
            "delta_mu": mu_fix - mu_ref,
            "chi2": fit_fixed_z.chisq,
            "ndof": fit_fixed_z.ndof,
        }
    )

if fit_free_z is not None:
    z_fit = fit_free_z.parameters[fit_free_z.param_names.index("z")]
    mu_fz, smu_fz = compute_mu(fit_free_z)
    mu_ref_fz = mu_lcdm(z_fit)
    print(f"\nFree-z fit:")
    print(f"  z_fit         = {z_fit:.5f} ± {fit_free_z.errors.get('z', np.nan):.5f}")
    print(f"  μ_obs         = {mu_fz:.4f} ± {smu_fz:.4f}")
    print(f"  μ_ΛCDM(z_fit) = {mu_ref_fz:.4f}")
    print(f"  Δμ            = {mu_fz - mu_ref_fz:+.4f}")
    results_summary.append(
        {
            "fit": "free_z",
            "z": z_fit,
            "z_source": "SALT3 photo-z",
            "x1": fit_free_z.parameters[fit_free_z.param_names.index("x1")],
            "c": fit_free_z.parameters[fit_free_z.param_names.index("c")],
            "mu": mu_fz,
            "sigma_mu": smu_fz,
            "mu_lcdm": mu_ref_fz,
            "delta_mu": mu_fz - mu_ref_fz,
            "chi2": fit_free_z.chisq,
            "ndof": fit_free_z.ndof,
        }
    )

summary_df = pd.DataFrame(results_summary)
summary_df

In [ ]:
# ── Hubble diagram with ΛCDM overlay ─────────────────────────────────────────
z_range = np.linspace(0.01, 1.5, 300)
mu_theo = np.array([mu_lcdm(z) for z in z_range])
mu_empty = 5.0 * np.log10((2.998e5 / H0) * z_range * (1 + z_range / 2)) + 25.0

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(z_range, mu_theo, "k-", lw=1.5, label="ΛCDM (Ωm=0.3, ΩΛ=0.7)")
ax.plot(z_range, mu_empty, "k--", lw=1.0, label="Empty universe")

for row in results_summary:
    label = f"{row['fit']}  z={row['z']:.4f}  μ={row['mu']:.3f}±{row['sigma_mu']:.3f}"
    ax.errorbar(
        row["z"],
        row["mu"],
        yerr=row["sigma_mu"] if not np.isnan(row["sigma_mu"]) else 0,
        fmt="o",
        markersize=10,
        label=label,
    )

ax.set_xlabel("Redshift z")
ax.set_ylabel("Distance modulus μ")
ax.set_title(f"Hubble diagram — {tns_name or 'diaObj ' + str(DIA_OBJECT_ID)}")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(FIGS_DIR / "hubble_diagram.png", dpi=150, bbox_inches="tight")
plt.show()

## 12 — Save results

In [ ]:
# Save summary to parquet
out_path = DATA_DIR / f"salt3_results_{DIA_OBJECT_ID}.parquet"
summary_df.to_parquet(out_path, index=False)
print(f"Results saved → {out_path}")

# Save src light curve
if not src_df.empty:
    src_path = DATA_DIR / f"src_{DIA_OBJECT_ID}.parquet"
    src_df.to_parquet(src_path, index=False)
    print(f"src saved     → {src_path}")

# Save fp light curve
if not fp_df.empty:
    fp_path = DATA_DIR / f"fp_{FINK_ID}.parquet"
    fp_df.to_parquet(fp_path, index=False)
    print(f"fp saved      → {fp_path}")

print("\n=== SUMMARY ===")
print(f"Object       : {tns_name or 'unknown'}  ({tns_type or 'unclassified'})")
print(f"diaObjectId  : {DIA_OBJECT_ID}")
print(f"Fink ID      : {FINK_ID}")
print(f"Redshift ref : {z_used}  ({z_source})")
if not summary_df.empty:
    print(summary_df[["fit", "z", "mu", "sigma_mu", "delta_mu", "chi2", "ndof"]].to_string(index=False))